# Import library

In [12]:
import os
import json
import time
import zipfile
import datetime
import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from tqdm import tqdm
from collections import defaultdict
import timm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [13]:
print("Available Vision Transformer Models: ")
timm.list_models("vit*")

Available Vision Transformer Models: 


['vit_base_mci_224',
 'vit_base_patch8_224',
 'vit_base_patch14_dinov2',
 'vit_base_patch14_reg4_dinov2',
 'vit_base_patch16_18x2_224',
 'vit_base_patch16_224',
 'vit_base_patch16_224_miil',
 'vit_base_patch16_384',
 'vit_base_patch16_clip_224',
 'vit_base_patch16_clip_384',
 'vit_base_patch16_clip_quickgelu_224',
 'vit_base_patch16_gap_224',
 'vit_base_patch16_plus_240',
 'vit_base_patch16_plus_clip_240',
 'vit_base_patch16_reg4_gap_256',
 'vit_base_patch16_rope_224',
 'vit_base_patch16_rope_ape_224',
 'vit_base_patch16_rope_mixed_224',
 'vit_base_patch16_rope_mixed_ape_224',
 'vit_base_patch16_rope_reg1_gap_256',
 'vit_base_patch16_rpn_224',
 'vit_base_patch16_siglip_224',
 'vit_base_patch16_siglip_256',
 'vit_base_patch16_siglip_384',
 'vit_base_patch16_siglip_512',
 'vit_base_patch16_siglip_gap_224',
 'vit_base_patch16_siglip_gap_256',
 'vit_base_patch16_siglip_gap_384',
 'vit_base_patch16_siglip_gap_512',
 'vit_base_patch16_xp_224',
 'vit_base_patch32_224',
 'vit_base_patch32_384'

In [14]:
from utilis import Utility, Data, Visualization, Score, save_config_and_lr, load_config_and_lr
from chunk_handle import get_chunk_indices, iter_chunks, compute_global_image_stats, compute_global_label_stats, save_transform_and_scaler_ViT, load_transform_and_scaler_ViT

/Users/viethuy/Working_space/Neurips/Neurips2025_Weak_lensing/utilis.py:352: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  DEVICE = "mps" if torch.has_mps else "cpu"


In [15]:
device="mps"

In [31]:
params = {
    'model': 'vitamin_xlarge_256',
    'im_size': 256,
    'inp_channels': 1,
    'device': device,
    'lr': 1e-4,
    'weight_decay': 1e-6,
    'batch_size': 101,
    'num_workers' : 0,
    'epochs': 10,
    'out_features': 4,
    'Balance_Dataset': False,
}

In [32]:
class AlienNet(nn.Module):
    def __init__(self, model_name=params['model'], out_features=params['out_features'],
                 inp_channels=params['inp_channels'], pretrained=True):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained,
                                       in_chans=inp_channels)
        if model_name.split('_')[0] == 'efficientnet':
            n_features = self.model.classifier.in_features
            self.model.conv_stem = nn.Conv2d(inp_channels, 40, kernel_size=(3, 3),
                                             stride=(2, 2), padding=(1, 1), bias=False)
            self.model.classifier = nn.Linear(n_features, out_features)
        
        elif model_name.split('_')[0] == 'nfnet':
            n_features = self.model.head.fc.in_features
            self.model.head.fc = nn.Linear(n_features, out_features)
            
        elif model_name.split('_')[0] == 'vit':
            n_features = self.model.head.in_features
            self.model.head = nn.Linear(n_features, out_features, bias=True)
    
    def forward(self, x):
        x = self.model(x)
        means = x[:, :2]
        log_sigmas = x[:, 2:]    # Predict log(σ) to ensure positivity
        sigmas = torch.exp(log_sigmas)
        return means, sigmas 
        # return x
    
    


# If the transform and scaler is avaiable, ignore that

In [33]:
transform, label_scaler = load_transform_and_scaler_ViT(256,transform_file='./side_module_ViT/transform_params.pkl', scaler_file='./side_module_ViT/label_scaler.pkl')

Loaded transform with Resize=(256, 256), Mean=-0.00016638042870908976, Std=0.02047532983124256
Loaded label scaler with Mean=[0.29021683 0.81345297], Std=[0.1055216  0.06600116]


# Model architecture

In [34]:
# Simple CNN architecture for parameter estimation

class Simple_CNN(nn.Module):
    def __init__(self, height, width, num_targets):
        super(Simple_CNN, self).__init__()
        # Convolutional layers
        self.conv_stack = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self._feature_size = self._get_conv_output_size(height, width)
        
        # Fully connected layers (regressor head)
        self.fc_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self._feature_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_targets)
        )

    def _get_conv_output_size(self, height, width):
        dummy_input = torch.zeros(1, 1, height, width)
        output = self.conv_stack(dummy_input)
        return int(np.prod(output.size()))

    def forward(self, x):
        x = self.conv_stack(x)
        x = self.fc_stack(x)
        means = x[:, :2]
        log_sigmas = x[:, 2:]    # Predict log(σ) to ensure positivity
        sigmas = torch.exp(log_sigmas)
        return means, sigmas     # Note that means and sigmas here have to be rescaled properly due to standardization

In [35]:
import torch
import torch.nn as nn
import torch.optim as optim
from vit_pytorch import ViT
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

from utilis import KL_div_posterior_loss, CosmologyDataset, train_epoch, validate_epoch
from Models import CustomViT, Simple_CNN

In [36]:
import os
import numpy as np
from typing import Optional, Tuple, Generator, Dict, Any
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
import time
from tqdm import tqdm
import pickle
import wandb  # Import Weights & Biases


def incremental_train(model: nn.Module, 
                      config,
                      optimizer: optim.Optimizer, 
                      criterion: nn.Module, 
                      chunk_dir: str,
                      split_ratio: float = 0.8,
                      epochs_per_chunk: int = 1,
                      device: str = 'mps' if torch.has_mps else 'cpu',
                      verbose: bool = False,
                      log_file: str = None,
                      config_file: str = None) -> Dict[str, Any]:
    """
    Incrementally train a PyTorch model across chunks without loading the entire dataset.
    
    - Loads one chunk at a time.
    - Splits each chunk into train/test sets.
    - Trains the model on the train set using batches (for memory efficiency within chunk).
    - Validates once per epoch using concatenated validation data from all chunks.
    - Logs training progress to a text file and Weights & Biases.
    - Uses tqdm for progress visualization.
    - Saves config and final learning rate in a single timestamped folder.
    - Returns a summary dict with loss history, etc.
    
    Args:
        model: PyTorch model (nn.Module).
        optimizer: PyTorch optimizer.
        criterion: Loss function (e.g., nn.MSELoss()).
        chunk_dir: Directory containing chunk files.
        split_ratio: Fraction for train split (e.g., 0.8).
        epochs_per_chunk: Number of epochs to train on each chunk.
        device: Device to train on ('cuda' or 'cpu').
        verbose: If True, print additional progress details.
        log_file: Path to the text file for logging training progress (set dynamically if None).
        config_file: Path to save the config and learning rate (set dynamically if None).
    
    Returns:
        Dict with 'train_losses' (list of lists: per-chunk losses), 'total_epochs', etc.
    """
    # Initialize Weights & Biases
    # Generate a fixed timestamp for this training run
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    
    # Set file paths using the fixed timestamp
    model_dir = f"./side_module_ViT/model_{timestamp}"
    if log_file is None:
        log_file = f"{model_dir}/training_log.txt"
    if config_file is None:
        config_file = f"{model_dir}/training_config.pkl"
    config.MODEL_SAVE_PATH = f"{model_dir}/best_model.pth"

    # Initialize log file (overwrite if exists)
    os.makedirs(os.path.dirname(log_file) or '.', exist_ok=True)
    with open(log_file, 'w') as f:
        f.write("Training Log\n")
        f.write(f"Started at {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Config: EPOCHS={config.EPOCHS}, BATCH_SIZE={config.BATCH_SIZE}, LEARNING_RATE={config.LEARNING_RATE}, DEVICE={config.DEVICE}, MODEL_SAVE_PATH={config.MODEL_SAVE_PATH}\n\n")

    device = config.DEVICE
    model.to(device)
    model.train()
    total_val_datasets = []
    
    indices = get_chunk_indices(chunk_dir)
    train_losses = []  # List of lists: losses per epoch per chunk
    val_losses = []  # List of validation losses per epoch  
    total_samples = 0
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    best_val_loss = float('inf')
    start_time = time.time()

    for epoch in tqdm(range(config.EPOCHS), desc="Epochs", leave=True):
        with open(log_file, 'a') as f:
            f.write(f"Epoch {epoch+1}/{config.EPOCHS}\n")
        epoch_train_losses = []
        
        for chunk_idx in tqdm(indices, desc="Chunks", leave=False):
            if verbose:
                print(f"Processing chunk {chunk_idx}...")
            with open(log_file, 'a') as f:
                f.write(f"  Processing chunk {chunk_idx}...\n")
            
            # Load single chunk (assumes it fits in memory)
            noisy_chunk, label_chunk, _ = next(iter_chunks(chunk_dir, indices=[chunk_idx]))
            
            # Split into train/test (stratified if labels are categorical; here simple split)
            Nsys = noisy_chunk.shape[1]
            if verbose:
                print(f"NP_idx : {Nsys}")
            NP_idx = np.arange(Nsys)   
            shape = noisy_chunk.shape[2:]
            if verbose:
                print('n samples in chunk (H,W):', shape)
            split_fraction = 1 - split_ratio
            seed = 113

            train_NP_idx, val_NP_idx = train_test_split(NP_idx, test_size=split_fraction, random_state=seed)

            noisy_kappa_train = noisy_chunk[:, train_NP_idx]      # shape = (Ncosmo, len(train_NP_idx), 1424, 176)
            label_train = label_chunk[:, train_NP_idx]         # shape = (Ncosmo, len(train_NP_idx), 5)
            noisy_kappa_val = noisy_chunk[:, val_NP_idx]          # shape = (Ncosmo, len(val_NP_idx), 1424, 176)
            label_val = label_chunk[:, val_NP_idx]             # shape = (Ncosmo, len(val_NP_idx), 5)

            Ntrain = label_train.shape[0] * label_train.shape[1]
            Nval = label_val.shape[0] * label_val.shape[1]
            if verbose:
                print(f'Shape of the split training data = {noisy_kappa_train.shape}')
                print(f'Shape of the split validation data = {noisy_kappa_val.shape}')
                print(f'Shape of the split training labels = {label_train.shape}')
                print(f'Shape of the split validation labels = {label_val.shape}')

            # Reshape the data for CNN
            X_train = noisy_kappa_train.reshape(Ntrain, *shape)
            X_val = noisy_kappa_val.reshape(Nval, *shape)

            # Keep only the first 2 cosmological parameters
            label_dim = label_train.shape[2]
            y_train = label_train.reshape(Ntrain, label_dim)[:, :2]
            y_val = label_val.reshape(Nval, label_dim)[:, :2]

            # Label standardization
            y_train_scaled = label_scaler.transform(y_train)
            y_val_scaled = label_scaler.transform(y_val)
            if verbose:
                print(f"Label stats (from train set): Mean={label_scaler.mean_}, Std={np.sqrt(label_scaler.var_)}")

            train_dataset = CosmologyDataset(data=X_train, labels=y_train_scaled, transform=transform)
            val_dataset = CosmologyDataset(data=X_val, labels=y_val_scaled, transform=transform)
            
            if verbose:
                if len(train_dataset) > 0:
                    train_sample, train_label = train_dataset[0]
                    print(f"Chunk {chunk_idx}: Train sample image shape: {train_sample.shape}, label shape: {train_label.shape}")
                if len(val_dataset) > 0:
                    val_sample, val_label = val_dataset[0]
                    print(f"Chunk {chunk_idx}: Val sample image shape: {val_sample.shape}, label shape: {val_label.shape}")
                
            
            if epoch == 0:
                total_val_datasets.append(val_dataset)

            train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)

            for epoch_chunk in range(epochs_per_chunk):
                train_loss = train_epoch(model, train_loader, criterion, optimizer, config.DEVICE)
                epoch_train_losses.append(train_loss)
                if verbose:
                    print(f"Epoch {epoch_chunk+1}/{epochs_per_chunk} | Train Loss: {train_loss:.6f}")
                with open(log_file, 'a') as f:
                    f.write(f"Epoch {epoch_chunk+1}/{epochs_per_chunk} | Train Loss: {train_loss:.6f}\n")
                # Log training loss to W&B
                wandb.log({"chunk": epoch*len(indices)+chunk_idx, "train_loss": train_loss})

        train_losses.append(epoch_train_losses)

        # Validate once per epoch using concatenated validation dataset
        if epoch == 0:
            all_val_data = np.concatenate([ds.data for ds in total_val_datasets], axis=0)
            all_val_labels = np.concatenate([ds.labels for ds in total_val_datasets], axis=0)
            concatenated_val_dataset = CosmologyDataset(
                data=all_val_data,
                labels=all_val_labels,
                transform=transform
            )
            val_loader = DataLoader(
                concatenated_val_dataset,
                batch_size=config.BATCH_SIZE,
                shuffle=False
            )

        val_loss = validate_epoch(model, val_loader, criterion, config.DEVICE)
        val_losses.append(val_loss) 
        scheduler.step(val_loss)
        # Log validation loss to W&B
        wandb.log({"epoch": epoch + 1, "val_loss": val_loss, "avg_train_loss": np.mean(epoch_train_losses)})
        with open(log_file, 'a') as f:
            f.write(f"Epoch {epoch+1}/{config.EPOCHS} | Avg Train Loss: {np.mean(epoch_train_losses):.6f} | Val Loss: {val_loss:.6f}\n")

        # Save the best model based on validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), config.MODEL_SAVE_PATH)
            with open(log_file, 'a') as f:
                f.write(f"  -> New best model saved to {config.MODEL_SAVE_PATH} (Val Loss: {val_loss:.6f})\n")
            # Log best model save event to W&B
            wandb.save(config.MODEL_SAVE_PATH)

    # Save config and final learning rate
    save_config_and_lr(config, optimizer, config_file)

    end_time = time.time()
    total_time_min = (end_time - start_time) / 60
    with open(log_file, 'a') as f:
        f.write(f"\nTraining finished in {total_time_min:.2f} minutes.\n")
    # Log total training time to W&B
    wandb.log({"total_training_time_min": total_time_min})

    model.load_state_dict(torch.load(config.MODEL_SAVE_PATH, weights_only=True))
    # Finish W&B run
    wandb.finish()
    return {
        'train_losses': train_losses,
        'total_val_datasets': total_val_datasets,
        'total_epochs': config.EPOCHS,
        'total_samples': total_samples,
        'val_losses': val_losses    
    }


# Training script


/var/folders/y7/lp_fcj6s2bn7220l1k8jhbh40000gn/T/ipykernel_38494/1507971631.py:23: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  device: str = 'mps' if torch.has_mps else 'cpu',


In [ ]:

class Config:
    IMG_HEIGHT = 1424
    IMG_WIDTH = 176
    
    # Parameters to predict (Omega_m, S_8, sigma_Omega_m, sigma_S_8)
    NUM_TARGETS = 4

    # Training hyperparameters
    BATCH_SIZE = 128
    EPOCHS = 2
    LEARNING_RATE = 2e-4
    WEIGHT_DECAY = 1e-4   # L2 regularization to prevent overfitting
    IMG_RESIZE = 256  # Resize images to 256x256 for ViT
    DEVICE = "mps" if torch.has_mps else "cpu"
    MODEL_SAVE_PATH = None  # Will be set dynamically with timestamp


config = Config()
# model = CustomViT()

model=AlienNet()
model_name="ViT_base_patch32_384"


# model=Simple_CNN(256, 256, config.NUM_TARGETS)
# model_name="Simple_CNN"


# model = CustomViT()
# model_name="Custom_ViT"

pretrain = False
previous_path = "side_module_ViT/model_20250918_153642"  # Set to the timestamp of the previous run if pretrain=True

if pretrain:
    if previous_path is None:
        raise ValueError("Please specify previous_timestamp for pretraining (e.g., '20250918_144500').")
    config_file = f'{previous_path}/training_config.pkl'
    config, last_lr = load_config_and_lr(config_file=config_file)
    model.load_state_dict(torch.load("side_module_ViT/model_20250918_153642/best_model.pth", weights_only=True))
    optimizer = optim.Adam(model.parameters(), lr=last_lr, weight_decay=config.WEIGHT_DECAY)
else:
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

criterion = KL_div_posterior_loss  # Replace with your criterion
config.EPOCHS = 10 # Set total epochs for continued training

run_name = f"{model_name}_run_{time.strftime('%Y_%m_%d_%H_%M_%S')}"

# Initialize W&B run
wandb.init(
    entity="huy_ben",
    project="cosmology_training",
          # project name (string, not f-string with stray bracket)
    name=run_name,   
    config={
        "IMG_HEIGHT": config.IMG_HEIGHT,
        "IMG_WIDTH": config.IMG_WIDTH,
        "NUM_TARGETS": config.NUM_TARGETS,
        "BATCH_SIZE": config.BATCH_SIZE,
        "EPOCHS": config.EPOCHS,
        "LEARNING_RATE": config.LEARNING_RATE,
        "WEIGHT_DECAY": config.WEIGHT_DECAY,
        "DEVICE": config.DEVICE,
        "IMG_RESIZE": config.IMG_RESIZE,
        "split_ratio": 0.8,
        "epochs_per_chunk": 1
})



# Continue training
history = incremental_train(
    model=model,
    config=config,
    optimizer=optimizer,
    criterion=criterion,
    chunk_dir='./dataset/chunk_kappa_noise_new',
    split_ratio=0.8,
    epochs_per_chunk=1,
    verbose=False
)

/var/folders/y7/lp_fcj6s2bn7220l1k8jhbh40000gn/T/ipykernel_38494/2926800407.py:14: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  DEVICE = "mps" if torch.has_mps else "cpu"


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epochs:   0%|          | 0/10 [00:14<?, ?it/s]


RuntimeError: MPS backend out of memory (MPS allocated: 16.85 GiB, other allocations: 386.45 MiB, max allowed: 22.64 GiB). Tried to allocate 6.00 GiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [ ]:
# # Training script
# config = Config()
# # model = Simple_CNN(config.IMG_HEIGHT, config.IMG_WIDTH, config.NUM_TARGETS)  # Replace with your model class

# model= CustomViT()

# pretrain = False
# previous_path = "side_module_ViT/model_20250918_153642"  # Set to the timestamp of the previous run if pretrain=True, e.g., '20250918_144500'

# if pretrain:
#     if previous_path is None:
#         raise ValueError("Please specify previous_timestamp for pretraining (e.g., '20250918_144500').")
#     config_file = f'{previous_path}/training_config.pkl'
#     config, last_lr = load_config_and_lr(config_file=config_file)
#     # model.load_state_dict(torch.load(config.MODEL_SAVE_PATH, weights_only=True))
#     model.load_state_dict(torch.load("side_module_ViT/model_20250918_153642/best_model.pth", weights_only=True))
#     optimizer = optim.Adam(model.parameters(), lr=last_lr, weight_decay=config.WEIGHT_DECAY)
# else:
#     optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

# criterion = KL_div_posterior_loss  # Replace with your criterion
# # Continue training
# result = incremental_train(
#     model=model,
#     config=config,
#     optimizer=optimizer,
#     criterion=criterion,
#     chunk_dir='./dataset/chunk_kappa_noise_new',
#     split_ratio=0.8,
#     epochs_per_chunk=1,
#     verbose=False
# )

In [ ]:
total_val_datasets = history['total_val_datasets']
train_losses = history['train_losses']
total_samples = history['total_samples']
total_epochs = history['total_epochs']
if total_val_datasets:
    all_val_data = np.concatenate([ds.data for ds in total_val_datasets], axis=0)
    all_val_labels = np.concatenate([ds.labels for ds in total_val_datasets], axis=0)
    concatenated_val_dataset = CosmologyDataset(
        data=all_val_data,
        labels=all_val_labels,
        transform=transform # Use transform from first dataset
    )
    val_loader = DataLoader(
        concatenated_val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False
    )
else:
    concatenated_val_loader = None
    print("No validation datasets to concatenate.")

In [ ]:
model.eval()
means_pred_list, sigmas_pred_list = [], []
pbar = tqdm(val_loader, total=len(val_loader), desc="Validating")
with torch.no_grad():
    for X, _ in pbar:
        X = X.to(config.DEVICE)
        means_pred, sigmas_pred = model(X)         
        means_pred_list.append(means_pred.cpu().numpy()) 
        sigmas_pred_list.append(sigmas_pred.cpu().numpy())
        
mean_val = np.concatenate(means_pred_list, axis=0)
mean_val = label_scaler.inverse_transform(mean_val)          # inverse transform

errorbar_val = np.concatenate(sigmas_pred_list, axis=0)
errorbar_val = errorbar_val*label_scaler.var_**0.5           # rescale by the training label std

In [ ]:
## Include the prior that the cosmological parameters are not negative
negative_mask = mean_val - errorbar_val < 0
errorbar_val[negative_mask] = mean_val[negative_mask]

In [ ]:
validation_score = Score._score_phase1(
    true_cosmo=all_val_labels,
    infer_cosmo=mean_val,
    errorbar=errorbar_val
)
print('averaged score:', np.mean(validation_score))
print('averaged error bar:', np.mean(errorbar_val, 0))

In [ ]:
# Comparison of the means & standard deviations of the posterior distributions and the validation labels
all_val_labels_inv = label_scaler.inverse_transform(all_val_labels)
plt.errorbar(all_val_labels_inv[:,0], mean_val[:,0], yerr=errorbar_val[:,0], 
             fmt='o', capsize=3, capthick=1, ecolor='grey')
plt.plot(sorted(all_val_labels_inv[:,0]), sorted(all_val_labels_inv[:,0]),
         color = 'grey', linestyle='dashed')
plt.xlim(np.min(all_val_labels_inv[:,0]), np.max(all_val_labels_inv[:,0]))
plt.ylim(0, 0.7)
plt.xlabel('Ground Truth')
plt.ylabel('Prediction')
plt.title(r'$\Omega_m$')
plt.show()

plt.errorbar(all_val_labels_inv[:,1], mean_val[:,1], yerr=errorbar_val[:,1], 
             fmt='o', capsize=3, capthick=1, ecolor='grey')
plt.plot(sorted(all_val_labels_inv[:,1]), sorted(all_val_labels_inv[:,1]),
         color = 'grey', linestyle='dashed')
plt.xlim(np.min(all_val_labels_inv[:,1]), np.max(all_val_labels_inv[:,1]))
plt.ylim(0.65, 1)
plt.xlabel('Ground Truth')
plt.ylabel('Prediction')
plt.title(r'$S_8$')
plt.show()

In [ ]:
# Initialize Data class object
data_obj = Data(data_dir="./dataset", USE_PUBLIC_DATASET=True)

# Load train data
data_obj.load_train_data()

# Load test data
data_obj.load_test_data()

In [ ]:
test_dataset = CosmologyDataset(
    data=data_obj.kappa_test, 
    transform=transform
)

test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

In [ ]:
test_dataset = CosmologyDataset(
    data=data_obj.kappa_test, 
    transform=transform
)

test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

In [ ]:
model.eval()
means_pred_list, sigmas_pred_list = [], []
pbar = tqdm(test_loader, total=len(test_loader), desc="Inference on the test set")
with torch.no_grad():
    for X in pbar:
        X = X.to(config.DEVICE)
        means_pred, sigmas_pred = model(X)         
        means_pred_list.append(means_pred.cpu().numpy()) 
        sigmas_pred_list.append(sigmas_pred.cpu().numpy())
        
mean = np.concatenate(means_pred_list, axis=0)
mean = label_scaler.inverse_transform(mean)          # inverse transform

errorbar = np.concatenate(sigmas_pred_list, axis=0)
errorbar = errorbar*label_scaler.var_**0.5           # rescale by the training label std

In [ ]:
## Include the prior that the cosmological parameters are not negative
negative_mask = mean - errorbar < 0
errorbar[negative_mask] = mean[negative_mask]

In [ ]:
data = {"means": mean.tolist(), "errorbars": errorbar.tolist()}
the_date = datetime.datetime.now().strftime("%y-%m-%d-%H-%M")
zip_file_name = 'Submission_' + the_date + '.zip'
zip_file = Utility.save_json_zip(
    submission_dir="submissions",
    json_file_name="result.json",
    zip_file_name=zip_file_name,
    data=data
)
print(f"Submission ZIP saved at: {zip_file}")

In [ ]:
# model.eval()
# means_pred_list, sigmas_pred_list = [], []
# pbar = tqdm(val_loader, total=len(val_loader), desc="Validating")
# with torch.no_grad():
#     for X, _ in pbar:
#         X = X.to(config.DEVICE)
#         means_pred, sigmas_pred = model(X)         
#         means_pred_list.append(means_pred.cpu().numpy()) 
#         sigmas_pred_list.append(sigmas_pred.cpu().numpy())
        
# mean_val = np.concatenate(means_pred_list, axis=0)
# mean_val = label_scaler.inverse_transform(mean_val)          # inverse transform

# errorbar_val = np.concatenate(sigmas_pred_list, axis=0)
# errorbar_val = errorbar_val*label_scaler.var_**0.5           # rescale by the training label std

In [ ]:
# model.eval()
# means_pred_list, sigmas_pred_list = [], []
# pbar = tqdm(val_loader, total=len(val_loader), desc="Validating")
# with torch.no_grad():
#     for X, _ in pbar:
#         X = X.to(config.DEVICE)
#         means_pred, sigmas_pred = model(X)         
#         means_pred_list.append(means_pred.cpu().numpy()) 
#         sigmas_pred_list.append(sigmas_pred.cpu().numpy())
        
# mean_val = np.concatenate(means_pred_list, axis=0)
# mean_val = label_scaler.inverse_transform(mean_val)          # inverse transform

# errorbar_val = np.concatenate(sigmas_pred_list, axis=0)
# errorbar_val = errorbar_val*label_scaler.var_**0.5           # rescale by the training label std

In [ ]:
plt.plot(history['val_losses'], label='Validation Loss')
plt.plot(np.mean(np.array(history['train_losses']), axis=1), label='Training Loss')
plt.legend()
plt.show()

In [ ]:
np.array(result['train_losses']).shape